# Import

In [ ]:
import os
import bm25s
import numpy as np

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from beir.retrieval.models import SentenceBERT

In [ ]:
dataset = "scifact"
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset)
out_dir = os.path.join(os.getcwd(), "datasets")
data_path = util.download_and_unzip(url, out_dir)

In [ ]:
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

# Dense Retrieval (Semantic)

In [ ]:
dense_model = DRES(SentenceBERT("sentence-transformers/all-MiniLM-L6-v2"), batch_size=64)
dense_retriever = EvaluateRetrieval(dense_model, score_function="cos_sim")
dense_results = dense_retriever.retrieve(corpus, queries)
dense_results

In [ ]:
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(qrels, dense_results, [10])
ndcg, _map, recall, precision

# Sparse Retrieval (BM25)

In [ ]:
corpus_ids = list(corpus.keys())
corpus_texts = [f"{corpus[cid].get('title', '')} {corpus[cid].get('text', '')}" for cid in corpus_ids]

query_ids = [qid for qid in queries.keys() if qid in qrels]
query_texts = [queries[qid] for qid in query_ids]
corpus_tokens = bm25s.tokenize(corpus_texts)

retriever = bm25s.BM25(method="lucene", k1=1.2, b=0.75)
retriever.index(corpus_tokens)

query_tokens = bm25s.tokenize(query_texts)
docs, scores = retriever.retrieve(query_tokens, corpus=corpus_ids, k=100)

sparse_results = {}
for i, qid in enumerate(query_ids):
    # docs[i] и scores[i] - это массивы длины k
    sparse_results[qid] = {docs[i][j]: float(scores[i][j]) for j in range(len(docs[i]))}

ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(qrels, sparse_results, [10])

ndcg, _map, recall, precision

In [ ]:
import numpy as np
import math
from collections import Counter

print("4. Extracting query meta-features (x_q) using pure Python IDF...")

# 1. Быстро считаем Document Frequency (DF) для корпуса (сколько раз слово встречается в разных документах)
df_dict = Counter()
for text in corpus_texts:
    # Используем set, так как для DF нам важно наличие слова в документе, а не его частота
    unique_tokens = set(text.lower().split())
    df_dict.update(unique_tokens)

N = len(corpus_texts)

# 2. Классическая формула IDF (как в Lucene/BM25)
def get_idf(token):
    df = df_dict.get(token, 0)
    # Если слова нет в корпусе (OOV), его IDF = 0
    return math.log(1 + (N - df + 0.5) / (df + 0.5)) if df > 0 else 0.0

query_features = {}

# 3. Собираем матрицу фичей
for qid, q_text in zip(query_ids, query_texts):
    tokens = q_text.lower().split()
    if not tokens:
        continue

    idfs = [get_idf(t) for t in tokens]

    if not idfs:
        idfs = [0.0]

    f1_len = len(tokens)                  # Длина запроса
    f2_max_idf = float(np.max(idfs))      # Редкость самого специфичного слова
    f3_mean_idf = float(np.mean(idfs))    # Средняя информативность запроса
    f4_var_idf = float(np.var(idfs))      # Дисперсия (есть ли перекос в сторону одного редкого слова)

    query_features[qid] = [f1_len, f2_max_idf, f3_mean_idf, f4_var_idf]

print(f"Success! Extracted features for {len(query_features)} queries.")

# Посмотрим на пример
sample_qid = list(query_features.keys())[0]
print(f"Sample query '{sample_qid}' features (Len, Max IDF, Mean IDF, Var IDF):")
print([round(f, 4) for f in query_features[sample_qid]])

In [ ]:
# ==========================================
# НЕДОСТАЮЩИЙ БЛОК: Нормализация и расчет NDCG
# ==========================================

# 1. Min-Max нормализация на уровне каждого отдельного запроса
def normalize_scores(results):
    normalized_results = {}
    for q_id, doc_scores in results.items():
        if not doc_scores:
            continue
        scores = list(doc_scores.values())
        min_score, max_score = min(scores), max(scores)

        normalized_results[q_id] = {}
        for d_id, score in doc_scores.items():
            if max_score > min_score:
                norm_score = (score - min_score) / (max_score - min_score)
            else:
                norm_score = 0.5
            normalized_results[q_id][d_id] = norm_score
    return normalized_results

print("Normalizing dense and sparse scores...")
norm_dense = normalize_scores(dense_results)
norm_sparse = normalize_scores(sparse_results)

# 2. Быстрая функция для подсчета NDCG одного запроса
def compute_ndcg_at_k(q_id, hybrid_scores, k=10):
    if q_id not in qrels:
        return 0.0

    # Сортируем документы по убыванию гибридного скора
    ranked_docs = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:k]

    dcg = 0.0
    for i, (d_id, _) in enumerate(ranked_docs):
        rel = qrels[q_id].get(d_id, 0)
        if rel > 0:
            dcg += (2**rel - 1) / np.log2(i + 2)

    # Считаем IDCG (идеальный порядок на основе ground truth)
    ideal_docs = sorted(qrels[q_id].items(), key=lambda x: x[1], reverse=True)[:k]
    idcg = 0.0
    for i, (_, rel) in enumerate(ideal_docs):
        if rel > 0:
            idcg += (2**rel - 1) / np.log2(i + 2)

    return dcg / idcg if idcg > 0 else 0.0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("Running Global Grid Search for Static Hybrid Baseline...")

alpha_grid = np.arange(0.0, 1.05, 0.05)
mean_ndcg_scores = []

# Проходим по всем значениям alpha от 0 до 1
for alpha in alpha_grid:
    query_ndcgs = []

    for q_id in queries.keys():
        if q_id not in qrels:
            continue

        candidate_docs = set(norm_dense.get(q_id, {}).keys()).union(set(norm_sparse.get(q_id, {}).keys()))

        # Считаем гибридный скор по формуле со скриншота
        hybrid_scores = {}
        for d_id in candidate_docs:
            s_dense = norm_dense.get(q_id, {}).get(d_id, 0.0)
            s_sparse = norm_sparse.get(q_id, {}).get(d_id, 0.0)
            hybrid_scores[d_id] = alpha * s_dense + (1 - alpha) * s_sparse

        # Используем функцию compute_ndcg_at_k, которую мы писали в предыдущем скрипте
        ndcg = compute_ndcg_at_k(q_id, hybrid_scores, k=10)
        query_ndcgs.append(ndcg)

    # Усредняем NDCG по всем запросам для текущего alpha
    mean_ndcg = np.mean(query_ndcgs) if query_ndcgs else 0.0
    mean_ndcg_scores.append(mean_ndcg)

# Ищем лучшее статическое значение
best_idx = np.argmax(mean_ndcg_scores)
best_static_alpha = alpha_grid[best_idx]
best_static_ndcg = mean_ndcg_scores[best_idx]

print(f"BM25 Only (alpha=0.0): NDCG@10 = {mean_ndcg_scores[0]:.4f}")
print(f"Dense Only (alpha=1.0): NDCG@10 = {mean_ndcg_scores[-1]:.4f}")
print(f"Best Static Hybrid (alpha={best_static_alpha:.2f}): NDCG@10 = {best_static_ndcg:.4f}")

# ==========================================
# Отрисовка графика для статьи
# ==========================================
plt.figure(figsize=(8, 5))
plt.plot(alpha_grid, mean_ndcg_scores, marker='o', markersize=5, linestyle='-', color='#1f77b4', linewidth=2, label='Static Hybrid Interpolation')

# Выделяем точку максимума
plt.plot(best_static_alpha, best_static_ndcg, marker='*', color='#d62728', markersize=12,
         label=f'Optimal Static $\\alpha$={best_static_alpha:.2f}\n(NDCG={best_static_ndcg:.4f})')

# Линии для наглядности
plt.axvline(x=best_static_alpha, color='#d62728', linestyle='--', alpha=0.6)
plt.axhline(y=best_static_ndcg, color='#d62728', linestyle='--', alpha=0.6)

# Настройка осей и легенды (в стиле научных статей)
plt.title("Effect of Static Interpolation Weight on Retrieval Performance", fontsize=12, pad=15)
plt.xlabel("$\\alpha$ (Weight of Dense Retriever)", fontsize=11)
plt.ylabel("Mean NDCG@10", fontsize=11)
plt.xticks(np.arange(0.0, 1.1, 0.1))
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(loc='lower center', fontsize=10)
plt.tight_layout()

# Сохраняем график в высоком разрешении для Overleaf
plt.savefig("static_alpha_benchmark.pdf", format='pdf', dpi=300)
plt.show()